In [ ]:
# CELL 1: imports and paths

import pandas as pd
import numpy as np
from pathlib import Path

# Assume this notebook is in `notebooks/` – go up one level to repo root
PROJECT_ROOT = Path("..").resolve()

DATA_CLEAN = PROJECT_ROOT / "00_data" / "clean"
DATA_PROCESSED = PROJECT_ROOT / "00_data" / "processed"

# States in our analysis
KEEP_STATES = ["FL", "GA", "AL", "SC", "WA", "CO", "ID", "OR"]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CLEAN DIR:", DATA_CLEAN)
print("PROCESSED DIR:", DATA_PROCESSED)

PROJECT_ROOT: /Volumes/T7/Opiod Project/pds-2025-opioids-straight-as
CLEAN DIR: /Volumes/T7/Opiod Project/pds-2025-opioids-straight-as/00_data/clean
PROCESSED DIR: /Volumes/T7/Opiod Project/pds-2025-opioids-straight-as/00_data/processed


In [ ]:
# CELL 2: helper functions


def std_state_abbrev(s: pd.Series) -> pd.Series:
    """Standardize state abbreviations (upper + strip)."""
    return s.astype(str).str.upper().str.strip()


def std_county(c: pd.Series) -> pd.Series:
    """Standardize county names so they match across datasets."""
    return (
        c.astype(str)
        .str.upper()
        .str.strip()
        .str.replace(" COUNTY", "", regex=False)
        .str.replace(" PARISH", "", regex=False)
        .str.replace(" BOROUGH", "", regex=False)
        .str.replace(" CENSUS AREA", "", regex=False)
    )

In [ ]:
# CELL 3: load county population (wide -> long)

pop_path = DATA_CLEAN / "county_population_2000_2015.csv"
pop_wide = pd.read_csv(pop_path)

print("Population columns:", list(pop_wide.columns))
print("Population shape BEFORE reshape:", pop_wide.shape)

# ------------------------------------------------------------------
# 1. Identify the year-specific columns (POPESTIMATEYYYY)
# ------------------------------------------------------------------
year_cols = [c for c in pop_wide.columns if c.startswith("POPESTIMATE")]
print("Year columns detected:", year_cols)

if not year_cols:
    raise ValueError("No POPESTIMATE* columns found – check the population file.")

# ------------------------------------------------------------------
# 2. Melt wide -> long: one row per county-year
# ------------------------------------------------------------------
id_vars = [
    "STATE",  # numeric state code
    "COUNTY",  # numeric county code
    "STNAME",  # state name
    "CTYNAME",  # county name
    "STATE_ABBR",  # two-letter state abbrev (this is gold)
    "FIPS",  # full county FIPS if present
]

id_vars = [c for c in id_vars if c in pop_wide.columns]  # keep only existing

pop_long = pop_wide.melt(
    id_vars=id_vars,
    value_vars=year_cols,
    var_name="year_var",
    value_name="population",
)

# Extract year from 'POPESTIMATEYYYY'
pop_long["year"] = pop_long["year_var"].str.extract(r"(\d{4})").astype(int)

# ------------------------------------------------------------------
# 3. Standardize to common column names: state, county, year, population
# ------------------------------------------------------------------
pop = pop_long.rename(
    columns={
        "STATE_ABBR": "state",
        "CTYNAME": "county",
    }
)

# If STATE_ABBR isn't there for some reason, fall back to STNAME
if "state" not in pop.columns:
    pop = pop.rename(columns={"STNAME": "state"})

# Keep just what we need
pop = pop[["state", "county", "year", "population"]].copy()

# ------------------------------------------------------------------
# 4. Standardize keys and filter years + states
# ------------------------------------------------------------------
pop["state"] = std_state_abbrev(pop["state"])
pop["county"] = std_county(pop["county"])
pop["year"] = pop["year"].astype(int)

# Keep project years and states
pop = pop[(pop["year"] >= 2006) & (pop["year"] <= 2015)]
pop = pop[pop["state"].isin(KEEP_STATES)].copy()

print("Population shape AFTER reshape + filter:", pop.shape)
pop.head()

Population columns: ['SUMLEV', 'REGION', 'DIVISION', 'STATE', 'COUNTY', 'STNAME', 'CTYNAME', 'ESTIMATESBASE2000', 'POPESTIMATE2001', 'POPESTIMATE2002', 'POPESTIMATE2003', 'POPESTIMATE2004', 'POPESTIMATE2005', 'POPESTIMATE2006', 'POPESTIMATE2007', 'POPESTIMATE2008', 'POPESTIMATE2009', 'CENSUS2010POP', 'POPESTIMATE2010', 'STATE_FIPS', 'COUNTY_FIPS', 'FIPS', 'POPESTIMATE2011', 'POPESTIMATE2012', 'POPESTIMATE2013', 'POPESTIMATE2014', 'POPESTIMATE2015', 'STATE_ABBR']
Population shape BEFORE reshape: (3112, 28)
Year columns detected: ['POPESTIMATE2001', 'POPESTIMATE2002', 'POPESTIMATE2003', 'POPESTIMATE2004', 'POPESTIMATE2005', 'POPESTIMATE2006', 'POPESTIMATE2007', 'POPESTIMATE2008', 'POPESTIMATE2009', 'POPESTIMATE2010', 'POPESTIMATE2011', 'POPESTIMATE2012', 'POPESTIMATE2013', 'POPESTIMATE2014', 'POPESTIMATE2015']
Population shape AFTER reshape + filter: (5220, 4)


,state,county,year,population
15560,AL,AUTAUGA,2006,51328
15561,AL,BALDWIN,2006,168121
15562,AL,BARBOUR,2006,27861
15563,AL,BIBB,2006,22099
15564,AL,BLOUNT,2006,55485


In [ ]:
# CELL 4: load mortality (df_overdose_cleaned.csv)

mort_path = DATA_CLEAN / "df_overdose_cleaned.csv"
mort = pd.read_csv(mort_path)

print("Mortality columns:", mort.columns)
print("Mortality shape BEFORE cleaning:", mort.shape)

# Rename columns to standard names
rename_map_mort = {
    "state": "state",
    "county": "county",
    "year": "year",
    "deaths": "drug_deaths",
}
mort = mort.rename(columns=rename_map_mort)

# Standardize
mort["state"] = std_state_abbrev(mort["state"])
mort["county"] = std_county(mort["county"])
mort["year"] = mort["year"].astype(int)

# Filter to 2006–2015 & project states
mort = mort[(mort["year"] >= 2006) & (mort["year"] <= 2015)]
mort = mort[mort["state"].isin(KEEP_STATES)].copy()

print("Mortality shape AFTER filter:", mort.shape)
mort.head()

Mortality columns: Index(['county', 'county_code', 'year', 'drug/alcohol_induced_cause',
       'drug/alcohol_induced_cause_code', 'deaths', 'state'],
      dtype='object')
Mortality shape BEFORE cleaning: (6300, 7)
Mortality shape AFTER filter: (1281, 7)


,county,county_code,year,drug/alcohol_induced_cause,drug/alcohol_induced_cause_code,drug_deaths,state
0,BALDWIN,1003,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,29.0,AL
1,CALHOUN,1015,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,12.0,AL
2,ETOWAH,1055,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,14.0,AL
3,HOUSTON,1069,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,10.0,AL
4,JEFFERSON,1073,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,80.0,AL


In [ ]:
# CELL 5: load cleaned ARCOS county-year aggregates

arcos_path = DATA_PROCESSED / "arcos_county_year.csv"
arcos = pd.read_csv(arcos_path)

print("ARCOS columns:", arcos.columns)
print("ARCOS shape BEFORE cleaning:", arcos.shape)

# Standardize keys
arcos["state"] = std_state_abbrev(arcos["state"])
arcos["county"] = std_county(arcos["county"])
arcos["year"] = arcos["year"].astype(int)

# Filter to project states and years
arcos = arcos[arcos["state"].isin(KEEP_STATES)].copy()
arcos = arcos[(arcos["year"] >= 2006) & (arcos["year"] <= 2015)]

print("ARCOS shape AFTER filter:", arcos.shape)
arcos.head()

ARCOS columns: Index(['state', 'county', 'year', 'total_pills', 'total_mme',
       'n_transactions'],
      dtype='object')
ARCOS shape BEFORE cleaning: (43822, 6)
ARCOS shape AFTER filter: (5086, 6)


,state,county,year,total_pills,total_mme,n_transactions
330,AL,AUTAUGA,2006,2850573.246,5.166274e+07,8838
331,AL,AUTAUGA,2007,3167354.666,5.309262e+07,9760
332,AL,AUTAUGA,2008,3453070.648,5.746818e+07,10389
333,AL,AUTAUGA,2009,3759035.404,5.830612e+07,10528
334,AL,AUTAUGA,2010,3841098.472,6.263222e+07,11549


In [ ]:
# CELL 6: build master county-year panel

# Start with population as backbone
panel = pop.copy()

# Merge ARCOS shipments
panel = panel.merge(
    arcos[["state", "county", "year", "total_pills", "total_mme", "n_transactions"]],
    on=["state", "county", "year"],
    how="left",
    validate="1:1",
)

# Merge mortality
panel = panel.merge(
    mort[["state", "county", "year", "drug_deaths"]],
    on=["state", "county", "year"],
    how="left",
    validate="1:1",
)

print("Panel shape after merges:", panel.shape)
panel[
    ["state", "county", "year", "population", "total_pills", "total_mme", "drug_deaths"]
].head()

Panel shape after merges: (5220, 8)


,state,county,year,population,total_pills,total_mme,drug_deaths
0,AL,AUTAUGA,2006,51328,2850573.246,5.166274e+07,NaN
1,AL,BALDWIN,2006,168121,8586086.494,1.628179e+08,11.0
2,AL,BARBOUR,2006,27861,1018381.979,1.169558e+07,NaN
3,AL,BIBB,2006,22099,962487.912,1.658575e+07,NaN
4,AL,BLOUNT,2006,55485,1590574.274,2.555605e+07,NaN


In [ ]:
# CELL 7: handle missing shipments + feature engineering basics

# Counties with no ARCOS data -> treat as 0 shipments
panel["total_pills"] = panel["total_pills"].fillna(0)
panel["total_mme"] = panel["total_mme"].fillna(0)
panel["n_transactions"] = panel["n_transactions"].fillna(0)

# Overdose deaths: keep NaN as "censored / missing" – your prof warned about this!
panel["missing_deaths"] = panel["drug_deaths"].isna().astype(int)

# --- per capita metrics ---
panel["mme_per_capita"] = panel["total_mme"] / panel["population"]
panel["pills_per_capita"] = panel["total_pills"] / panel["population"]
panel["overdoses_per_100k"] = (panel["drug_deaths"] / panel["population"]) * 100_000

# log transforms (add 1 to avoid log(0))
panel["log_mme_per_capita"] = np.log1p(panel["mme_per_capita"])
panel["log_overdoses_per_100k"] = np.log1p(panel["overdoses_per_100k"])

panel[["state", "county", "year", "mme_per_capita", "overdoses_per_100k"]].head()

,state,county,year,mme_per_capita,overdoses_per_100k
0,AL,AUTAUGA,2006,1006.521665,NaN
1,AL,BALDWIN,2006,968.456784,6.542907
2,AL,BARBOUR,2006,419.783224,NaN
3,AL,BIBB,2006,750.520260,NaN
4,AL,BLOUNT,2006,460.593891,NaN


In [ ]:
# CELL 8: treatment + control groups + DiD helpers

# Which rows are from FL vs WA?
panel["is_FL"] = (panel["state"] == "FL").astype(int)
panel["is_WA"] = (panel["state"] == "WA").astype(int)

# Group labels for plotting
FL_CONTROLS = ["GA", "AL", "SC"]
WA_CONTROLS = ["CO", "ID", "OR"]

panel["group_FL"] = np.where(
    panel["state"] == "FL",
    "treated_FL",
    np.where(panel["state"].isin(FL_CONTROLS), "control_FL", "other"),
)

panel["group_WA"] = np.where(
    panel["state"] == "WA",
    "treated_WA",
    np.where(panel["state"].isin(WA_CONTROLS), "control_WA", "other"),
)

# Generic treated + post indicators
panel["treated"] = panel["state"].isin(["FL", "WA"]).astype(int)

# Policy years
FL_POLICY_YEAR = 2010
WA_POLICY_YEAR = 2012

panel["post"] = np.where(
    ((panel["state"] == "FL") & (panel["year"] >= FL_POLICY_YEAR))
    | ((panel["state"] == "WA") & (panel["year"] >= WA_POLICY_YEAR)),
    1,
    0,
)

# Event time relative to policy (for treated states only)
panel["t_minus_policy"] = np.where(
    panel["state"] == "FL",
    panel["year"] - FL_POLICY_YEAR,
    np.where(panel["state"] == "WA", panel["year"] - WA_POLICY_YEAR, np.nan),
)

# Fixed-effects helpers
panel["county_fe"] = panel["state"] + "_" + panel["county"]
panel["year_fe"] = panel["year"]

panel[
    [
        "state",
        "county",
        "year",
        "group_FL",
        "group_WA",
        "treated",
        "post",
        "t_minus_policy",
    ]
].head()

,state,county,year,group_FL,group_WA,treated,post,t_minus_policy
0,AL,AUTAUGA,2006,control_FL,other,0,0,NaN
1,AL,BALDWIN,2006,control_FL,other,0,0,NaN
2,AL,BARBOUR,2006,control_FL,other,0,0,NaN
3,AL,BIBB,2006,control_FL,other,0,0,NaN
4,AL,BLOUNT,2006,control_FL,other,0,0,NaN


In [ ]:
# CELL 9: save master panel

out_csv = DATA_PROCESSED / "master_county_year_panel.csv"
out_parquet = DATA_PROCESSED / "master_county_year_panel.parquet"

panel.to_csv(out_csv, index=False)
panel.to_parquet(out_parquet, index=False)

print("Saved:")
print(" -", out_csv)
print(" -", out_parquet)
panel.shape

Saved:
 - /Volumes/T7/Opiod Project/pds-2025-opioids-straight-as/00_data/processed/master_county_year_panel.csv
 - /Volumes/T7/Opiod Project/pds-2025-opioids-straight-as/00_data/processed/master_county_year_panel.parquet


(5220, 23)